# Step 02: Exploratory Data Analysis (hints visible)

> This is the **open** version: hints and early findings are shown inline.
> For the version with hints hidden, use `02_eda.ipynb`.

**Goal:** EDA here is not "make every chart". Every section answers a question whose answer
**changes a decision** in this project. Write the decision implication under each section.

| Section | Decision it feeds |
|---|---|
| 1. Sanity check | Is the data loaded correctly? |
| 2. Duplicates & templates | Step 04 split strategy (leakage) |
| 3. Text length | Training `max_seq_length`, cost estimate |
| 4. Label balance | How to sample the ~3k tickets for labelling |
| 5. Confusable queues | Final category list (ADR-008) |
| 6. Priority | Priority levels (ADR-008) |
| 7. Entities & placeholders | `entities` field in the schema |
| 8. Agent answers | `suggested_reply` length limit |
| 9. Version effects | Whether to filter by dataset version |

Rules: `SEED = 42` everywhere; read raw examples, not just numbers.

In [ ]:
import re

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_colwidth", 200)
SEED = 42

df = pd.read_json("../data/raw/tickets_en.jsonl", lines=True)
df.shape

## 1. Sanity check
Look at `df.head()`, `df.dtypes`, `df.isna().sum()`.
- Are there empty subjects (`""`)? What share?
- Does every row have a queue, priority and type?

**Implication:** _..._

In [ ]:
# your code

## 2. Duplicates & templates
Exact duplicates were already removed (4,497 of them!). Now look for **near-duplicates**.
- How many `body` values are duplicated when you ignore the subject? (`df["body"].duplicated()`)
- What are the 10 most common **first lines** of `body`? (`df["body"].str.split("\n").str[0].value_counts()`)
- Normalise text (lowercase, collapse whitespace, strip punctuation) and count duplicates again.
- Bonus: are duplicated `answer` values a sign of templated tickets?

**Why it matters:** if near-copies end up in both train and test, test scores lie (leakage).

**Implication:** _..._

In [ ]:
# your code

## 3. Text length
Plot a histogram of `df["text"].str.len()` and of approx tokens (`len / 4`).
- Do length distributions differ by queue? (`df.groupby("queue")["text"].str.len()` won't work; create a `len` column first, then `groupby(...).describe()`)
- Training input = system prompt (~350 tokens) + ticket + JSON output (~300 tokens). What
  `max_seq_length` covers 99% of tickets?

**Implication:** _..._

In [ ]:
# your code

## 4. Label balance
Bar charts of `queue`, `priority`, `type` (`value_counts().plot.barh()`).
- If we sample 3,000 tickets **randomly**, how many would the smallest queue get?
- Compare: random vs stratified vs capped-per-class sampling. Which gives the small model
  enough examples of rare classes without distorting reality too much?

**Implication:** _..._

In [ ]:
# your code

## 5. Confusable queues (most important section)
Technical Support vs IT Support vs Product Support vs Customer Service: are they really different?
1. Read 5 random tickets from each of those queues (`df[df.queue == q].sample(5, random_state=SEED)`).
   Cover the label and guess. How often are you right?
2. Find the words that distinguish each queue:
   ```python
   from sklearn.feature_extraction.text import TfidfVectorizer
   # fit on df["text"], then average the TF-IDF rows per queue and print the top 15 terms
   ```
3. Train a quick baseline: `TfidfVectorizer` + `LogisticRegression`, then
   `classification_report` and `confusion_matrix` on queue. **Which queues get confused with which?**

The baseline is also a **benchmark**: if a fine-tuned LLM can't beat TF-IDF + logistic
regression on `category`, that's important to know (and to say in interviews).

**Implication (candidate final category list):** _..._

In [ ]:
# your code

## 6. Priority
- English has only low / medium / high. Our schema has 4 levels including `urgent`. Options:
  drop `urgent`, or let the teacher assign `urgent` for genuine emergencies. Which do you prefer and why?
- `pd.crosstab(df.queue, df.priority, normalize="index")`: how much is priority "explained"
  by queue alone? Read 5 `high` and 5 `low` tickets: is the difference visible in the text?
- Same baseline as section 5 but predicting priority: how hard is this task?

**Implication:** _..._

In [ ]:
# your code

## 7. Entities & placeholders
Our schema extracts `order_id`, `product`, `account_email`, `amount`.
- Count placeholder tokens: `df["text"].str.findall(r"<[a-z_]+>").explode().value_counts()`
- How often do order numbers, emails, or money amounts actually appear? Try regexes such as
  `r"\$\s?\d"`, `r"\b[\w.-]+@[\w.-]+\.\w+\b"`, `r"(?i)order\s*#?\s*\w*\d"`.
- Is there HTML (`<br>`) to clean?

**Question:** are our 4 entity fields the right ones for *this* data? If a field is almost always
null, the model learns "always null", which is useless. What would you replace it with?

**Implication:** _..._

In [ ]:
# your code

## 8. Agent answers
- Length distribution of `answer` (chars and approx tokens).
- Read 5 answers: tone, structure, do they promise things (refunds, timelines)?

**Implication:** is `max_length=1200` for `suggested_reply` right? Should the teacher's reply
style imitate these answers?

In [ ]:
# your code

## 9. Version effects
`version` is 400, 52, 51 or missing.
- Do queue / priority distributions differ by version? (`pd.crosstab(df.version.fillna(-1), df.queue, normalize="index")`)
- Are text lengths different?

**Implication:** keep all versions, or filter?

In [ ]:
# your code

## Summary → copy into `docs/steps/02-dataset.md` Findings
1. Near-duplicates:
2. Max sequence length:
3. Sampling strategy:
4. Final categories:
5. Priority levels:
6. Entity fields:
7. Reply length:
8. Versions: